# Provenance Tracking

(This notebook demonstrates the provenance tracking feature in ProbPipe.)

Provenance tracking allows distributions to carry structured metadata about their origin, forming a DAG of probabilistic operations. This transforms distributions from anonymous outputs into traceable probabilistic objects.

### Key Concepts:

- **Provenance**
- **Operation (.op)**
- **Parents**
- **Details** (operation-specific metadata)
- **UID** (Unique identifier for each provenance node)

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from probpipe import Gaussian, EmpiricalDistribution, Provenance

np.random.seed(42)


## Example 1: Basic Provenance Tracking

Let's start by creating a simple distribution with provenance metadata.

In [ ]:
# Gaussian prior with provenance
prior = Gaussian(mean=np.array([0.0, 0.0]), cov=np.eye(2))._with_source(
    op="prior",
    name="theta",
    distribution_type="Gaussian",
    dimension=2
)

print("Distribution:", prior)
print("\nProvenance:", prior.source)
print("\nProvenance details:")
print(f"Operation: {prior.source.op}")
print(f"Details: {dict(prior.source.details)}")
print(f"Number of parents: {len(prior.source.parents)}")
print(f"Unique ID: {prior.source.uid}")

Distribution: Gaussian(mu = [0. 0.], cov = [[1. 0.]
 [0. 1.]])

Provenance: Provenance(op='prior' (name=theta, distribution_type=Gaussian, dimension=2), 0 parents)

Provenance details:
Operation: prior
Details: {'name': 'theta', 'distribution_type': 'Gaussian', 'dimension': 2}
Number of parents: 0
Unique ID: 11585d0f8c004ca8863e7355467a58eb


## Example 2: Chain of Operations

A common workflow in probabilistic programming:

1. **Prior** -> Create a parametric distribution (Gaussian)
2. **Approximate** -> Convert to sample-based representation (EmpiricalDistribution)
3. **Convert** -> Transform back to parametric form (Gaussian)

Each step creates provenance metadata that links to the previous step.

In [ ]:
# Step 1: Creating a Gaussian prior
print("Step 1- Creating Gaussian prior")
prior = Gaussian(mean=np.zeros(2), cov=np.eye(2))._with_source(
    op="prior",
    name="theta",
    distribution_type="Gaussian"
)
print(f"  Prior mean: {prior.mean}")
print(f"  Provenance op: '{prior.source.op}'")

Step 1- Creating Gaussian prior...
  Prior mean: [0. 0.]
  Provenance op: 'prior'


In [13]:
# Step 2: Approximate with empirical samples
print("Step 2- Approximating with empirical samples...")
emp = EmpiricalDistribution.from_distribution(prior, num_samples=4096)
print(f"Empirical distribution: {emp}")
print(f"Provenance op: '{emp.source.op}'")
print(f"Number of samples: {emp.source.details['num_samples']}")
print(f"Parent operation: '{emp.source.parents[0].op}'")

Step 2- Approximating with empirical samples...
Empirical distribution: EmpiricalDistribution(n=4096, dim=2, uniform, mean=[0.01954, 0.009711], std=[1.006, 0.9908])
Provenance op: 'approximate'
Number of samples: 4096
Parent operation: 'prior'


In [14]:
# Step 3: Convert back to parametric Gaussian
print("Step 3- Converting back to parametric Gaussian...")
approx_gauss = Gaussian.from_distribution(emp, num_samples=2048, method="moment_match")
print(f"Gaussian mean: {approx_gauss.mean}")
print(f"Provenance op: '{approx_gauss.source.op}'")
print(f"Conversion method: '{approx_gauss.source.details['method']}'")
print(f"Number of parents: {len(approx_gauss.source.parents)}")

Step 3- Converting back to parametric Gaussian...
Gaussian mean: [-0.00011836  0.00541625]
Provenance op: 'convert'
Conversion method: 'moment_match'
Number of parents: 1


### Inspecting the Provenance Chain

We can trace back through all operations that led to the final distribution:

In [15]:
# View the provenance chain
print("Provenance Chain:")
chain = approx_gauss.source.chain()
for i, node in enumerate(chain):
    print(f"{i}) {node.op} - {dict(node.details)}")

Provenance Chain:
0) convert - {'method': 'moment_match', 'num_samples': 2048, 'source_class': 'EmpiricalDistribution', 'target_class': 'Gaussian'}
1) approximate - {'method': 'empirical', 'num_samples': 4096, 'target_class': 'EmpiricalDistribution'}
2) prior - {'name': 'theta', 'distribution_type': 'Gaussian'}


### Visualizing the Provenance Tree

The `tree_repr()` method provides a hierarchical view of the provenance graph:

In [16]:
print("Provenance Tree:")
print(approx_gauss.source.tree_repr())

Provenance Tree:
└─ convert (method=moment_match, num_samples=2048, source_class=EmpiricalDistribution, target_class=Gaussian) [4e843b97]
  └─ approximate (method=empirical, num_samples=4096, target_class=EmpiricalDistribution) [e9403798]
    └─ prior (name=theta, distribution_type=Gaussian) [70d2771e]



### Operations available:

- `"prior"`: Initial distribution creation
- `"approximate"`: Converting parametric -> sample-based
- `"convert"`: Converting sample-based -> parametric


### API Methods:

- `dist._with_source(op, parents=[], **details)`: Attach provenance
- `dist.source`: Access provenance metadata
- `dist.source.chain()`: Get linearized provenance history
- `dist.source.tree_repr()`: Get hierarchical tree representation

### Next Steps:

- Integrate with TensorFlow Probability (TFP) distributions
- Add visualization tools for provenance DAGs
- Support serialization/deserialization of provenance
- Extend to full probabilistic program tracing